# 文字資料的統計

對應 DLI：06 NLP。

本課使用小型合成資料或 scikit-learn 內建資料，在 CPU 執行，不需下載原課程資料。依序執行所有儲存格。

[在 Google Colab 執行本課](https://colab.research.google.com/github/Samuel-ncu/AI-Stastic/blob/main/notebooks/10_text_statistics.ipynb) · [閱讀完整背景講義](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/10-text-statistics.md)

建議順序：先讀概念與手算 → 回答檢核 → 執行程式 → 修改練習。

[背景導讀：從統計到 AI 的模型設計](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/statistics-to-ai.md)

## 學習目標與先備知識

先讀機率分布與分類。本課從數詞開始理解文字如何變成向量，再介紹熵及相似度；不要求先會語言模型。



## 1. 文字怎麼變成模型輸入？

**來源摘要 [S8](https://scikit-learn.org/stable/modules/feature_extraction.html)：**詞袋表示把文字切成 token、計數並形成數值特徵。單詞詞袋不保留詞序，因此有明顯限制。

本教材自編兩句「ai helps students」與「students learn ai」。建立詞彙表 [ai,helps,learn,students] 後：

| 句子 | ai | helps | learn | students |
|---|---:|---:|---:|---:|
| ai helps students | 1 | 1 | 0 | 1 |
| students learn ai | 1 | 0 | 1 | 1 |

每列是一個文件向量。若對所有文字先建立詞彙表再切分，測試資料資訊可能進入前處理；正式分類要在訓練文字 fit，再對其他集合 transform。本課的小表只演示表示法，沒有做測試效能評估。

中文不一定有空白分詞。切詞、字元 n-gram 或模型 tokenizer 是不同選擇，會影響後續頻率及向量，不能直接假設英文示範可原封不動套用。



## 2. 詞頻如何變成機率分布？

本教材自編文件「ai ai data model」，總詞數 4。p(ai)=2/4，p(data)=p(model)=1/4。三個比例加總為 1。

這個分布是「在此文件中隨機抽一個詞位置」的經驗詞頻分布，不是某個語言模型對下一個詞的預測分布。先說清楚隨機的是什麼，熵才有意義。



## 3. 熵：平均有多少不確定性？

**來源摘要 [SC2](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.entropy.html)：**Shannon 熵以 $-\sum p_i\log p_i$ 定義；對數底數決定單位。SciPy 可依指定底數計算，也可計算兩個分布的相對熵。

使用底數 2 時：

$$H=-\sum_i p_i\log_2 p_i$$

本教材手算 [0.5,0.25,0.25]：

- 第一項：−0.5×(−1)=0.5。
- 另外兩項各為 −0.25×(−2)=0.5。
- 總和 H=1.5 bits。

若只有一種詞，p=1，H=0。若四種詞等比例，H=2 bits。這些值描述指定分布的不確定性，**不等於文章品質、知識價值或作者創意**。對 p=0 的項，按極限慣例記為 0，不直接計算 0×log(0)。



## 4. 餘弦相似度是怎麼算的？

對非零向量 u、v：

$$\cos(u,v)=\frac{u\cdot v}{\|u\|\|v\|}$$

點積是對應元素相乘後加總，長度是平方和開根號。本教材自編 u=[1,1,0]、v=[1,0,1]：點積為 1，兩者長度都是 $\sqrt2$，所以相似度為 0.5。

它比較方向。將同一計數向量全部乘 2，相似度仍為 1，不代表文件內容完全相同。零向量的數學公式分母為 0，實作需要明確處理。



## 5. 如何連到 DLI NLP？

先把「token、詞彙表、序列、向量」分清楚，再看 embedding 與序列模型。詞袋可當輕量基準，但「狗追貓」與「貓追狗」若只看相同詞的次數，可能得到相同表示；語意角色已被省略。

本課 entropy 是詞頻熵。第五課分類交叉熵是用真實標籤評估預測機率；語言模型下一詞交叉熵又有序列條件。公式有關，但資料、機率與評估問題不同。

## 補充｜從文字向量看懂 Transformer 的殘差連接

前面已把一段文字表示成向量。Transformer 會保留每個 token 的向量，並反覆更新它。以簡化的張量形狀 $[B,T,D]$ 為例，$B$ 是批次筆數，$T$ 是 token 數，$D$ 是每個 token 的表示維度。設 $h$ 為子層的輸入，$F(h)$ 是注意力或前饋網路算出的更新。相加前兩個張量的形狀一致。

$$h_{\rm sum}=h+F(h)$$

**自編手算。** 某個 token 原本的三維表示是 $[2,1,-1]$，前饋子層輸出 $[0.3,-0.2,0.1]$。相加得 $[2.3,0.8,-0.9]$。三個座標都是內部特徵，程式可直接相加。訓練時會以最終任務損失更新子層參數，$F(h)$ 的每個座標通常沒有教師直接提供的答案。

| 問題 | 統計迴歸的殘差 | Transformer 的殘差連接 |
|---|---|---|
| 差值或更新從哪來？ | $r_i=y_i-\hat y_i$，需有實測答案 | $F(h)$ 由子層從當前輸入算出 |
| 用在哪裡？ | 診斷預測偏差，或透過平方、絕對值等轉成損失 | 在前向運算中把內部特徵的更新加到輸入 |
| 量的單位 | 與目標 $y$ 相同，例如毫秒 | 與隱藏表示相容，例如每個 token 的 $D$ 個座標 |
| 推論時能否取得？ | 取得實測答案後才可計算 | 只要有輸入，子層就能計算並相加 |

兩者的數學關連是**以一個基準表達差量**。迴歸的關係可重排為 $y_i=\hat y_i+r_i$。殘差學習則把想學的映射寫成 $H(h)=h+F(h)$，相當於以輸入 $h$ 當基準，讓分支表示 $H(h)-h$。這裡的 $H$ 是想表達的層映射。教學可從同一個延遲例子理解「原先估計＋一段修正」，再回到表格辨認哪一段依賴標籤。參見 [第 05 課迴歸殘差](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/05-statistical-models-loss.md)。

**為什麼有旁路？** 在相加處，梯度可同時經過直接路徑和子層，對 $h$ 的局部導數含恆等項。若相加後還有正規化，梯度仍會經過該正規化。只憑一個加號無法推斷整個模型一定收斂或保證梯度大小。

**把四種運算放回架構。** 原始 Transformer 的前饋子層用兩個線性變換，中間放 ReLU。殘差連接把子層輸出加回輸入，再做 LayerNorm，亦即 $\mathrm{LN}(h+F(h))$，常稱 Post-LN。另一種設計把 LN 放在子層入口，寫作 $h+F(\mathrm{LN}(h))$，常稱 Pre-LN。這兩式省略 Dropout。於是 ReLU 負責前饋非線性，殘差旁路負責加總，LN 調整表示的尺度，最終 token 交叉熵衡量模型給正確 token 的機率。查閱特定模型時，應以其架構或程式確認所用啟動函數和正規化位置。

**自我檢核。** 如果 $F(h)=0$，加總點的結果是什麼？若原始 Transformer 此處使用 Post-LN，整個子層的輸出呢？答案依序是 $h$ 和 $\mathrm{LN}(h)$。這也解釋了為何要區分加總點和整個子層。

[來源：ResNet 原論文](https://arxiv.org/abs/1512.03385) · [Transformer 原論文 §3.1、§3.3、§3.4](https://arxiv.org/html/1706.03762v7#S3) · [Xiong 等人對 Pre-LN 與 Post-LN 的研究](https://proceedings.mlr.press/v119/xiong20b.html)

可回到 [第 06 課比較啟動函數](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/06-overfitting-generalization.md)，再從模型規格確認前饋層的啟動函數與正規化位置。

## 自我檢核

1. 分布 [0.5,0.5] 的熵是多少 bits？
2. u=[1,0] 與 v=[2,0] 的餘弦相似度是多少？
3. 兩句話詞袋一樣，就能斷定語意一樣嗎？

### 參考解答

1. 1 bit。
2. 1；兩者方向相同。
3. 不能。詞序、語法及語境可能不同。

## 環境

若套件尚未安裝，先執行下一格。安裝需網路，之後的實驗不需下載資料。

In [ ]:
%pip -q install numpy matplotlib scikit-learn

### 手算對照殘差相加與 Post-LN

In [ ]:
import numpy as np

# One batch, one token, three hidden features.
hidden = np.array([[[2., 1., -1.]]])
update = np.array([[[0.3, -0.2, 0.1]]])
assert hidden.shape == update.shape
combined = hidden + update
np.testing.assert_allclose(combined, [[[2.3, 0.8, -0.9]]])

# The original Post-LN puts normalization after the sum.
# This demonstration uses gamma=1, beta=0 and no dropout.
centered = combined - combined.mean(axis=-1, keepdims=True)
post_ln = centered / np.sqrt(combined.var(axis=-1, keepdims=True) + 1e-5)
print("Residual sum:", combined.round(3))
print("After LayerNorm:", post_ln.round(3))
np.testing.assert_allclose(post_ln.mean(axis=-1), 0, atol=1e-12)

In [ ]:
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
texts = ['ai helps students learn', 'students learn statistics', 'ai ai ai ai']
for text in texts:
    counts = Counter(text.split())
    p = np.array(list(counts.values()), dtype=float)
    p /= p.sum()
    print(counts, 'entropy bits=', -(p*np.log2(p)).sum())
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)
print('Vocabulary:', vectorizer.get_feature_names_out())
print('Cosine similarity:', cosine_similarity(X))
assert np.allclose(np.diag(cosine_similarity(X)), 1)

## 如何解讀

重複單一詞的經驗分布熵為 0，但不能因此宣稱內容毫無資訊。此為詞頻分布熵，不是已訓練語言模型的交叉熵。範例使用空白分詞英文；中文需另外決定斷詞策略。

## 練習

建立使用不同字詞但意思相近的兩句話，觀察詞袋方法的限制。

提交修改後的程式、結果圖表及 200 字解讀，列出至少一項限制。若用 AI 協助，說明你如何驗證結果。

## 連結原課程

回到 [DLI Notebook](https://github.com/NVDLI/fundamentals-of-deep-learning-zh/tree/main/course_content/tutorials)，找出本概念對應的資料處理、模型訓練或評估步驟。

## 出處與閱讀建議

- **S8**：[scikit-learn，Feature extraction（Bag of Words）](https://scikit-learn.org/stable/modules/feature_extraction.html)。
- **SC2**：[SciPy，scipy.stats.entropy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.entropy.html)。

本課以所列來源的概念作繁體中文摘要，並加入自行編寫的情境、手算、程式及練習；不是原網站的逐段翻譯。標為「來源摘要」者為改寫，標為「自編」者為本教材教學設計；短引文另以引號及「中譯」標明。來源連結可供對照原文，查閱日期：2026-11-17。
